In [1]:
# https://humans-to-robots-motion.github.io/mogaze/getting_started

In [2]:
from humoro.trajectory import Trajectory

full_traj = Trajectory()
full_traj.loadTrajHDF5("../../input/mogaze/p1_1_human_data.hdf5")

In [3]:
print("The data has dimension timeframe, state_size:")
print(full_traj.data.shape)
print("")
print("This is a list of jointnames (from the urdf) corresponding to the state dimensions:")
print(list(full_traj.description))
print("")
print("Some joints are used for scaling the human and do not change over time")
print("They are available in a dictionary:")
print(full_traj.data_fixed)

The data has dimension timeframe, state_size:
(53899, 66)

This is a list of jointnames (from the urdf) corresponding to the state dimensions:
['baseTransX', 'baseTransY', 'baseTransZ', 'baseRotX', 'baseRotY', 'baseRotZ', 'pelvisRotX', 'pelvisRotY', 'pelvisRotZ', 'torsoRotX', 'torsoRotY', 'torsoRotZ', 'neckRotX', 'neckRotY', 'neckRotZ', 'headRotX', 'headRotY', 'headRotZ', 'linnerShoulderRotX', 'linnerShoulderRotY', 'linnerShoulderRotZ', 'lShoulderRotX', 'lShoulderRotY', 'lShoulderRotZ', 'lElbowRotX', 'lElbowRotY', 'lElbowRotZ', 'lWristRotX', 'lWristRotY', 'lWristRotZ', 'rinnerShoulderRotX', 'rinnerShoulderRotY', 'rinnerShoulderRotZ', 'rShoulderRotX', 'rShoulderRotY', 'rShoulderRotZ', 'rElbowRotX', 'rElbowRotY', 'rElbowRotZ', 'rWristRotX', 'rWristRotY', 'rWristRotZ', 'lHipRotX', 'lHipRotY', 'lHipRotZ', 'lKneeRotX', 'lKneeRotY', 'lKneeRotZ', 'lAnkleRotX', 'lAnkleRotY', 'lAnkleRotZ', 'lToeRotX', 'lToeRotY', 'lToeRotZ', 'rHipRotX', 'rHipRotY', 'rHipRotZ', 'rKneeRotX', 'rKneeRotY', 'rKneeRo

In [4]:
from humoro.player_pybullet import Player
pp = Player()
pp.spawnHuman("Human1")
pp.addPlaybackTraj(full_traj, "Human1")

pybullet build time: Jan 29 2025 23:16:28


In [5]:
pp.showFrame(3000)

In [6]:
pp.play(duration=360, startframe=3000)

In [7]:
pp.spawnHuman("Human2", color=[0., 1., 0., 1.])
# this extracts a subtrajectory from the full trajectory:
sub_traj = full_traj.subTraj(3000, 3360)
# we change the startframe of the sub_traj,
# thus the player will play it at a different time:
sub_traj.startframe = 4000
pp.addPlaybackTraj(sub_traj, "Human2")
pp.play(duration=360, startframe=4000)

In [8]:
from humoro.load_scenes import autoload_objects
obj_trajs, obj_names = autoload_objects(pp, "../../input/mogaze/p1_1_object_data.hdf5", "../../input/mogaze/scene.xml")
pp.play(duration=360, startframe=3000)

In [9]:
from humoro.gaze import load_gaze
gaze_traj = load_gaze("../../input/mogaze/p1_1_gaze_data.hdf5")
pp.addPlaybackTrajGaze(gaze_traj)

(53900, 3)


In [10]:
pp.play(duration=360, startframe=3000)

In [11]:
print("calibration rotation quaternion:")
print(gaze_traj.data_fixed['calibration'])

calibration rotation quaternion:
[-0.31145353  0.37690775 -0.56556629  0.66413253]


In [12]:
pp.play_controls("../../input/mogaze/p1_1_segmentations.hdf5")

['null', 'cup_red', 'plate_blue', 'jug', 'plate_green', 'plate_red', 'cup_green', 'cup_blue', 'cup_pink', 'plate_pink', 'bowl']

null:
  count: 57 (0.504)
  time sum: 283.6 (0.633)
  time avg (std dev) [min/max]: 5.0 (5.3) [1.4/35.7]

cup_red:
  count: 11 (0.097)
  time sum: 29.6 (0.066)
  time avg (std dev) [min/max]: 2.7 (0.8) [1.0/3.9]

plate_blue:
  count: 4 (0.035)
  time sum: 13.7 (0.031)
  time avg (std dev) [min/max]: 3.4 (0.6) [2.5/4.2]

jug:
  count: 4 (0.035)
  time sum: 11.3 (0.025)
  time avg (std dev) [min/max]: 2.8 (0.3) [2.3/3.2]

plate_green:
  count: 4 (0.035)
  time sum: 13.1 (0.029)
  time avg (std dev) [min/max]: 3.3 (0.3) [2.9/3.6]

plate_red:
  count: 9 (0.080)
  time sum: 27.2 (0.061)
  time avg (std dev) [min/max]: 3.0 (0.4) [2.6/3.7]

cup_green:
  count: 6 (0.053)
  time sum: 16.3 (0.036)
  time avg (std dev) [min/max]: 2.7 (0.5) [1.9/3.2]

cup_blue:
  count: 6 (0.053)
  time sum: 17.2 (0.038)
  time avg (std dev) [min/max]: 2.9 (0.6) [2.0/3.6]

cup_pink:
  co

In [13]:
import h5py
with h5py.File("../mogaze/p1_1_segmentations.hdf5", "r") as segfile:
    # print first 5 segments:
    for i in range(5):
        print(segfile["segments"][i])

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '../mogaze/p1_1_segmentations.hdf5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
from humoro.kin_pybullet import HumanKin
kinematics = HumanKin()
kinematics.set_state(full_traj, 100)  # set state at frame 100
print("position of right wrist:")
wrist_id = kinematics.inv_index["rWristRotZ"]
pos = kinematics.get_position(wrist_id)
print(pos)